# Data-Generating Process

## The True Data-Generating Process

We assume there exists an underlying, unknown true data-generating process that yields pairs of feature vectors $X$ and target labels $Y$. This process is formally described by an unknown true probability distribution $p^*(X, Y)$ defined over a joint feature space $\mathcal{X}$ and target space $\mathcal{Y}$. The asterisk in $p^*$ denotes this true (and unknown) distribution, as opposed to any model or estimate of it.

Throughout, uppercase letters ($X$, $Y$) denote random variables, while lowercase letters ($x$, $y$, $x_i$, $y_i$) denote specific observed values.

Every time we collect data from the real world, we are **drawing a sample** $(x_i, y_i) \sim p^*(X, Y)$ from the true probability distribution. A sample is a concrete, realized instance of the pair of random variables $(X, Y)$.

For example, our data-generating process could be collecting images of animals and labeling them according to the animal class. In this case:

* **Observation:** The real-world photo captured by a camera.
* **Feature Vector ($x_i \in \mathcal{X}$):** The numerical representation of that observation (e.g., an $H \times W \times 3$ array of RGB pixel intensity values, flattened into a single vector).
* **Label ($y_i \in \mathcal{Y}$):** The assigned target output (e.g., the species name $\text{"Dog"}$).

Note that both the feature space $\mathcal{X}$ and the target space $\mathcal{Y}$ are entirely general. They can be discrete (e.g., classification labels), continuous (e.g., real numbers $\mathbb{R}$ in regression), vector-valued (e.g., axis-aligned 3D bounding box coordinates in $\mathbb{R}^6$), or high-dimensional structured outputs (e.g., text sequences or pixel masks).

## Why Model Data Generation as a Probability Distribution?

It might seem unintuitive to model the true data-generating process as a probability distribution. In our animal classification example, when we take a photograph of an animal, the label associated with that specific observation is deterministic - a cat is a cat, not a random variable.

If the label of an individual sample is deterministic, it might seem more intuitive to use a deterministic mapping function $y = f(x)$ to describe the relationship between the spaces $\mathcal{X}$ and $\mathcal{Y}$. In practice, however, using a joint probability distribution $p^*(X, Y)$ (and its conditional distribution $p^*(Y \mid X)$) is necessary for two main reasons:

* **Identical Observations Can Yield Different Labels Across Samples:** Even if every sample has a definitive ground truth in reality, two *different* draws from the data-generating process can produce the same feature vector $x$ while having different true labels:
  - *Observation Noise & Corruption:* Environmental artifacts erase distinguishing traits. A severely overexposed image of a white cat can produce an essentially indistinguishable pixel array from a severely overexposed image of a white dog.
  - *Label Noise:* Human annotators or automated pipelines disagree or make mistakes, so the recorded label $y$ may differ from the true label.
  - *Information Bottlenecks (Unobserved Variables):* Even clean, uncorrupted features may simply not contain the information needed to determine the label. For example, if the features only describe an animal's color histogram, a ginger cat and a ginger dog may have identical features. Similarly, an animal mostly hidden behind a fence may show too little of its body to be identified.

  Because $x$ alone cannot uniquely determine $y$ across all samples in these cases, the conditional distribution $p^*(Y \mid X = x)$ must express a probability distribution over candidate labels (e.g., $p^*(Y=\text{"Cat"} \mid X=x) = 0.9$ and $p^*(Y=\text{"Dog"} \mid X=x) = 0.1$).
* **Deterministic Mappings Are a Special Case:** Even in a noise-free setup where every feature vector $x$ uniquely determines its label via a function $y = f(x)$, probabilistic modeling remains valid. A fully deterministic mapping is simply a **degenerate (point-mass) probability distribution** that places all of its probability on a single label:

  $$p^*(Y = y \mid X = x) = \mathbb{1}[y = f(x)]$$

## Empirical Datasets as Approximations

In practice, we never have access to the underlying probability distribution $p^*(X, Y)$ directly. Instead, we collect a finite dataset $\mathcal{D}$ consisting of $N$ samples that we assume are **independent and identically distributed (i.i.d.)** draws from the true joint distribution of the data-generating process:

$$\mathcal{D} = \{(x_1, y_1), (x_2, y_2), \dots, (x_N, y_N)\}, \qquad (x_i, y_i) \overset{\text{i.i.d.}}{\sim} p^*(X, Y), \quad i = 1, \dots, N$$

The i.i.d. assumption is an idealization. Real-world data often violates it, for example when samples are correlated (several photos of the same animal) or when the distribution changes between training and deployment (*distribution shift*). Many practical failures of machine learning systems can be traced back to such violations.

We can construct the empirical distribution $\hat{p}_{\text{data}}(x, y)$, which places an equal probability weight of $\frac{1}{N}$ at the exact location of each observed data point:

$$\hat{p}_{\text{data}}(x,y) = \frac{1}{N} \sum_{i=1}^N \delta(x - x_i)\,\delta(y - y_i)$$

Here $\delta$ is the Dirac delta function. For discrete variables, $\delta(x - x_i)\,\delta(y - y_i)$ is simply the indicator $\mathbb{1}[x = x_i,\, y = y_i]$: it is 1 when $(x, y)$ equals the dataset pair $(x_i, y_i)$ and 0 otherwise. For continuous variables, the Dirac delta is an infinitely narrow spike with total area 1, so each data point contributes a probability mass of $\frac{1}{N}$.

A finite dataset is only a discrete collection of points, so $\hat{p}_{\text{data}}$ never becomes a smooth density. However, it still approximates $p^*$ in the sense that matters for learning: by the Law of Large Numbers, for any (integrable) function $g$ (e.g. a loss function), averages over the dataset converge to expectations under the true distribution:

$$\mathbb{E}_{\hat{p}_{\text{data}}}[g(X, Y)] = \frac{1}{N}\sum_{i=1}^N g(x_i, y_i) \xrightarrow{N \to \infty} \mathbb{E}_{p^*}[g(X, Y)]$$

In other words, $\hat{p}_{\text{data}}$ converges to $p^*$ *in distribution* (weakly). This is what justifies replacing expectations under the unknown $p^*$ with averages over the training set, which is the basis of empirical risk minimization.

## Factorizing the Joint Distribution

By the chain rule of probability, the joint distribution of the data-generating process $p^*(X, Y)$ can be factored in two mathematically equivalent ways:

$$p^*(X, Y) = p^*(Y \mid X)\, p^*(X) = p^*(X \mid Y)\, p^*(Y)$$

Each factorization tells a different, equally valid story of how a sample $(x, y)$ comes about. In our animal example:

* **Features first, then the label: $p^*(X)\, p^*(Y \mid X)$.**
  - $p^*(X)$ describes how likely we are to encounter each possible photo, before anyone looks at which animal it shows.
  - $p^*(Y \mid X)$ is the distribution over species for a specific photo $x$, e.g., $p^*(Y = \text{"Dog"} \mid X = x)$. This is the probabilistic label from the previous section.
* **Label first, then the features: $p^*(Y)\, p^*(X \mid Y)$.**
  - $p^*(Y)$ describes how common each species is among the photos we collect (e.g., the fraction of dog photos). It is called the *class prior*.
  - $p^*(X \mid Y)$ describes what photos of a given species look like. For $Y = \text{"Dog"}$, it is the distribution over all possible photos of dogs.

These two factorizations correspond to the two main families of machine learning models. **Discriminative models** learn only $p^*(Y \mid X)$: how to predict the label from the features. **Generative models** learn $p^*(X \mid Y)$ and $p^*(Y)$ (or $p^*(X)$ directly): what the data itself looks like. We will see how both kinds of models are trained when we discuss [maximum likelihood estimation](https://github.com/aecins/tutorials/blob/main/ml/maximum_likelihood_estimation.ipynb).